# the obliterated lane · unsloth QLoRA

Fine-tune an abliterated model on the constellation's world set made plain: unsloth 4-bit QLoRA, the mix = the world jsonl (uploaded via colab-mcp) + a public instruct set. Export both MLX and GGUF.

In [ ]:
!pip install unsloth[x] --no-deps 2>/dev/null; pip show unsloth >/dev/null 2>&1 || pip install unsloth[x]

In [ ]:
from unsloth import FastLanguageModel

# THE PICK — the lane's headline (swap the repo for the other three)
model_name = "OBLITERATUS/Ornith-1.5-9B-OBLITERATED"
max_seq = 4096
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq,
    dtype=None,
    load_in_4bit=True,
)

In [ ]:
from datasets import load_dataset

# THE MIX: the world set (uploaded) + a public instruct set
world = load_dataset("json", data_files="world_dataset.jsonl", split="train")
instruct = load_dataset("allenai/tulu-2-sft-mixture", split="train").select_columns(
    ["instruction", "output"]
) if False else load_dataset("databricks/databricks-dolly-15k", split="train")

def fmt(ex):
    return [
        {"role": "system", "content": ex.get("system", "you are the 8b-is engine.")},
        {"role": "user", "content": ex.get("instruction", "the keeper folds.")},
        {"role": "assistant", "content": ex.get("output", "admissible.")},
    ]

world = world.map(lambda ex: {"messages": fmt(ex)})
instruct = instruct.map(lambda ex: {"messages": fmt(ex)})
dataset = world.select_columns(["messages"])  # + instruct if the world needs neighbours

In [ ]:
from unsloth import UnslothTrainer, UnslothTrainingArguments, FastLanguageModel

model = FastLanguageModel.get_peft_model(model, r=16, target_modules=[
    "q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"
])
trainer = UnslothTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=UnslothTrainingArguments(
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        max_steps=120, learning_rate=2e-4, weight_decay=0.0,
        warmup_steps=10, lr_scheduler_type="linear",
        logging_steps=1, output_dir="out/lora",
    ),
)
trainer.train()

In [ ]:
# export the merged fp16 — the door to BOTH formats
FastLanguageModel.save_pretrained_merged(
    "./out/merged_fp16", tokenizer, save_method="merged_16bit",
)

In [ ]:
# GGUF: unsloth's own exporter, then the imatrix pass with the CORPUS
FastLanguageModel.save_pretrained_gguf(
    "./out/gguf", tokenizer, quantization_method="q4_k_m",
)
# !llama-imatrix -m ./out/gguf/**/q4_k_m.gguf -f corpus.txt -o imatrix.dat
# and re-quantize with the imatrix for the shelf (Q4_K_M/Q5_K_M)

In [ ]:
# MLX: the merge is plain fp16 — mlx-lm converts it for the Mac
# !pip install mlx-lm
# !python -m mlx_lm.convert -q -b 64 --hf-path ./out/merged_fp16 -o ./out/mlx

Done — download `world_dataset.jsonl` out + the two exports back through the colab-mcp proxy, seat the GGUF into the sidecar mirror, and the Mac reads the MLX room.